# 2. Databricks: Ossie to Metric View and back

This notebook reads the Ossie file from Snowflake, builds a Unity Catalog Metric
View, queries it, adds a measure, and exports Ossie again for the return trip.

Prerequisites: upload three files into this notebook's workspace folder:
`ossie_from_snowflake.yaml`, `customers.csv`, and `orders.csv`. The notebook reads
them from its own folder (`os.getcwd()`).

Names: this notebook uses catalog `demos` and schema `semantic_interop` to match
the Snowflake database and schema. Matching names means the table references in the
Ossie file work on both sides without rewriting.

## Step 1 - Install the Apache Ossie Databricks converter

Pure Python with PyYAML, pinned to a specific commit. If the cluster has no internet, upload the `ossie_converter/` folder to the workspace folder instead and add it to `sys.path`.

In [0]:
%pip install "git+https://github.com/apache/ossie.git@01058aa416423cf43a74e7f9fb7f5f70981a418e#subdirectory=converters/databricks"

  Cloning https://github.com/apache/ossie.git (to revision 01058aa416423cf43a74e7f9fb7f5f70981a418e) to /tmp/pip-req-build-823587b5
  Running command git clone --filter=blob:none --quiet https://github.com/apache/ossie.git /tmp/pip-req-build-823587b5
  Running command git rev-parse -q --verify 'sha^01058aa416423cf43a74e7f9fb7f5f70981a418e'
  Running command git fetch -q https://github.com/apache/ossie.git 01058aa416423cf43a74e7f9fb7f5f70981a418e
  Running command git checkout -q 01058aa416423cf43a74e7f9fb7f5f70981a418e
  Resolved https://github.com/apache/ossie.git to commit 01058aa416423cf43a74e7f9fb7f5f70981a418e
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for apache-ossie-databricks: fi

In [0]:
dbutils.library.restartPython()

## Step 2 - Set names

Set `CATALOG` and `SCHEMA` in the next cell to the same names as the Snowflake
database and schema (in notebook 1 those are `DEMOS` and `SEMANTIC_INTEROP`). When
the names match, the table references inside the Ossie file resolve here as is, and
the source rewrite later in the notebook does nothing.

If you cannot use matching names, for example if you cannot create a catalog named
`demos`, set `CATALOG` and `SCHEMA` to a catalog and schema you can use, and set
`SF_NAMESPACE` to the database and schema you used in Snowflake. The notebook then
rewrites the source names from the Snowflake namespace to yours when it builds the
Metric View, and back again on the return trip. Matching names keeps it simple; the
rewrite is the fallback when they differ.

In [0]:
import os
# Match these to the Snowflake database and schema (in notebook 1: DEMOS / SEMANTIC_INTEROP).
CATALOG = "demos"
SCHEMA  = "semantic_interop_2"
# SF_NAMESPACE is the database.schema you used in Snowflake. Keep it equal to
# CATALOG.SCHEMA when the names match. If you must use different names here, leave
# SF_NAMESPACE set to the Snowflake names so the source rewrite can align them.
SF_NAMESPACE  = "DEMOS.SEMANTIC_INTEROP_2"
DBX_NAMESPACE = f"{CATALOG}.{SCHEMA}"
FOLDER  = os.getcwd()   # files live in this notebook's workspace folder
METRIC_VIEW   = f"{CATALOG}.{SCHEMA}.sales_metric_view"
print("folder:", FOLDER)
print("metric view:", METRIC_VIEW)

folder: /Workspace/Users/bfegan@gmail.com/demos/interoperable_semantics
metric view: demos.semantic_interop_2.sales_metric_view


## Step 3 - Create the tables from CSV

Reads `customers.csv` and `orders.csv` with pandas and writes Delta tables with
explicit integer types, so the numbers match Snowflake exactly. Creating the
catalog needs the CREATE CATALOG privilege on the metastore. If you cannot create a
catalog named `demos`, set `CATALOG` above to one you can use and keep the schema
name.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

cust_schema = StructType([StructField('customer_id', IntegerType()),
                          StructField('customer_name', StringType()),
                          StructField('region', StringType())])
ord_schema = StructType([StructField('order_id', IntegerType()),
                         StructField('customer_id', IntegerType()),
                         StructField('order_amount', IntegerType()),
                         StructField('order_qty', IntegerType())])

cust_pd = pd.read_csv(f'{FOLDER}/customers.csv')
ord_pd  = pd.read_csv(f'{FOLDER}/orders.csv')
spark.createDataFrame(cust_pd, schema=cust_schema).write.mode('overwrite').saveAsTable(f'{CATALOG}.{SCHEMA}.customers')
spark.createDataFrame(ord_pd,  schema=ord_schema ).write.mode('overwrite').saveAsTable(f'{CATALOG}.{SCHEMA}.orders')
print('tables created')

tables created


In [0]:
display(spark.sql(f"""
  SELECT region, SUM(order_amount) total_amount, COUNT(order_id) order_count, SUM(order_qty) total_qty
  FROM {CATALOG}.{SCHEMA}.orders JOIN {CATALOG}.{SCHEMA}.customers USING (customer_id)
  GROUP BY region ORDER BY region"""))
# expect EAST 750/5/12, WEST 700/5/11

region,total_amount,order_count,total_qty
EAST,750,5,12
WEST,700,5,11


## Step 4 - The spec-version and dialect shim

Snowflake has adopted Ossie natively. Reading and writing the format are built-in
system functions (`SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW` and
`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML`), and they track spec version 0.1.1.
Databricks does not offer native Ossie functions yet, so on this side the conversion
runs through the open-source Apache converter, which tracks a later draft
(0.2.0.dev0) and expects a slightly different layout.

That gap in platform support is why the Databricks half of the demo carries a converter and this shim, while the
Snowflake half just calls a function.

The shim reconciles three differences without changing the meaning of the model:

- version tag: 0.1.1 from Snowflake, 0.2.0.dev0 for the converter.
- dialect label: SNOWFLAKE, versus ANSI_SQL or DATABRICKS.
- where metrics live: inside each dataset in Snowflake's output, versus a model-level list for the converter.

The reverse function (`converter_to_snowflake`) does more than swap those labels, and
it is the piece that makes the return trip land cleanly. A Databricks Metric View
does not carry Snowflake's separation of facts and dimensions, and the forward trip
dropped the fact columns from the model. So on the way back the shim rebuilds the
fact columns as fields, re-qualifies each measure column with its table (for example
`SUM(ORDERS.order_qty)`), and marks the joined columns as dimensions. Snowflake's
importer requires all three and rejects the file without them. The function reads a
measure's columns whether they arrive bare or already qualified, so a measure you add
here in Databricks comes back complete.

In [0]:
# Licensed under Apache-2.0 (this file is original to the demo, not from apache/ossie).
"""Bridge between Snowflake's Ossie dialect and the Apache Ossie Databricks converter.

Why this exists
---------------
Snowflake's SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW emits Ossie **0.1.1** and the
vendored Apache converter tracks **0.2.0.dev0** (an exact-match check). The two spec
revisions differ in three concrete ways that this module reconciles:

1. version string           0.1.1                 <->  0.2.0.dev0
2. expression dialect       SNOWFLAKE             <->  ANSI_SQL / DATABRICKS
3. metric placement         dataset custom_ext    <->  model-level `metrics`

Item (3) is the load-bearing one: Snowflake stores metrics inside
`datasets[*].custom_extensions[SNOWFLAKE].data` as a JSON blob, while the Apache
converter reads a top-level `metrics` list. Without hoisting, the generated Metric
View has no measures at all.

The transforms are deliberately narrow and reversible so the interop story stays
honest: the semantic content (names, expressions, relationships) is untouched; only
the envelope (version tag, dialect label, metric location) is adapted.
"""

import json
import re

import yaml

CONVERTER_OSSIE_VERSION = "0.2.0.dev0"   # what the vendored Apache converter requires
SNOWFLAKE_OSSIE_VERSION = "0.1.1"        # what Snowflake emits / expects on import
SNOWFLAKE_DIALECT = "SNOWFLAKE"
ANSI_DIALECT = "ANSI_SQL"
DATABRICKS_DIALECT = "DATABRICKS"


def _relabel_dialects(expression_obj, frm, to):
    """Relabel every `dialect: <frm>` to `<to>` inside an Ossie expression object."""
    if not isinstance(expression_obj, dict):
        return
    for d in expression_obj.get("dialects", []) or []:
        if d.get("dialect") == frm:
            d["dialect"] = to


def snowflake_to_converter(ossie_yaml, drop_fact_fields=True):
    """Snowflake Ossie 0.1.1  ->  Apache-converter-ready Ossie 0.2.0.dev0.

    - bumps the version string
    - relabels SNOWFLAKE-dialect expressions to ANSI_SQL (the converter only reads
      DATABRICKS/ANSI_SQL)
    - hoists metrics out of each dataset's SNOWFLAKE custom_extension up to a
      model-level `metrics` list, stripping the `<dataset>.` qualifier so measure
      expressions are bare fact columns (the Databricks idiom: SUM(order_amount))
    - by default drops fact fields (those with no `dimension` marker) so they do not
      become groupable Metric View dimensions; they live on inside measure expressions
    """
    root = yaml.safe_load(ossie_yaml)
    root["version"] = CONVERTER_OSSIE_VERSION

    for model in root.get("semantic_model", []) or []:
        hoisted = []
        for ds in model.get("datasets", []) or []:
            ds_name = ds.get("name", "")
            qual = re.compile(re.escape(ds_name) + r"\.", re.IGNORECASE)

            # Hoist metrics from this dataset's SNOWFLAKE custom_extension.
            kept_ext = []
            for ext in ds.get("custom_extensions", []) or []:
                if ext.get("vendor_name") == SNOWFLAKE_DIALECT:
                    blob = json.loads(ext.get("data") or "{}")
                    for m in blob.get("metrics", []) or []:
                        expr = qual.sub("", m["expr"])  # SUM(orders.order_amount) -> SUM(order_amount)
                        hoisted.append({
                            "name": m["name"],
                            "expression": {
                                "dialects": [{"dialect": ANSI_DIALECT, "expression": expr}]
                            },
                        })
                else:
                    kept_ext.append(ext)
            if kept_ext:
                ds["custom_extensions"] = kept_ext
            else:
                ds.pop("custom_extensions", None)

            # Fields: relabel dialects, strip field-level SNOWFLAKE extensions, and
            # optionally drop facts (kept only if they carry a `dimension` marker).
            new_fields = []
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
                f.pop("custom_extensions", None)
                if drop_fact_fields and "dimension" not in f:
                    continue
                new_fields.append(f)
            if new_fields:
                ds["fields"] = new_fields
            else:
                ds.pop("fields", None)

        if hoisted:
            model["metrics"] = (model.get("metrics", []) or []) + hoisted

    return yaml.safe_dump(root, sort_keys=False)


def converter_to_snowflake(ossie_yaml, dialect=SNOWFLAKE_DIALECT, model_name=None):
    """Apache-converter Ossie 0.2.0.dev0  ->  Snowflake-importable Ossie 0.1.1.

    Databricks Metric Views don't carry Snowflake's fact/dimension distinction, and the
    forward trip dropped the fact columns, so this reverse trip must rebuild what
    Snowflake's importer needs:

    - resets the version string and relabels DATABRICKS-dialect expressions to SNOWFLAKE
    - qualifies each measure's bare columns with the fact table (COUNT(order_id) ->
      COUNT(ORDERS.order_id)); Snowflake derived metrics need a logical-table-qualified
      column
    - reconstructs the referenced fact columns as fields on the fact dataset (no
      `dimension` marker => facts) so the metric expressions resolve
    - marks every field on a non-fact (joined) dataset with `dimension: {}` so Snowflake
      classifies region/customer_name as dimensions, not facts
    - optionally renames the model (the new semantic view name) without touching the
      fact dataset name
    """
    root = yaml.safe_load(ossie_yaml)
    root["version"] = SNOWFLAKE_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        datasets = model.get("datasets", []) or []
        fact_ds_name = datasets[0]["name"] if datasets else None

        # Fields: relabel dialects; mark joined-dataset fields as dimensions.
        for ds in datasets:
            is_fact = ds.get("name") == fact_ds_name
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), DATABRICKS_DIALECT, dialect)
                if not is_fact:
                    f.setdefault("dimension", {})

        # Metrics: relabel, qualify bare columns with the fact table, then collect
        # every fact column the metric references - whether it arrived bare
        # (COUNT(order_id)) or already qualified (SUM(ORDERS.order_qty)) - so the
        # fact fields can be rebuilt for all of them.
        fact_cols = []
        ref_re = re.compile(re.escape(fact_ds_name) + r"\.([A-Za-z_]\w*)") if fact_ds_name else None
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), DATABRICKS_DIALECT, dialect)
            if not fact_ds_name:
                continue
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" in d:
                    d["expression"] = _qualify_columns(d["expression"], fact_ds_name)
                    for c in ref_re.findall(d["expression"]):
                        if c not in fact_cols:
                            fact_cols.append(c)

        # Rebuild the referenced columns as fact fields on the fact dataset.
        if fact_ds_name and fact_cols:
            fact_ds = datasets[0]
            existing = {f["name"].lower() for f in fact_ds.get("fields", []) or []}
            flds = fact_ds.setdefault("fields", [])
            for c in fact_cols:
                if c.lower() not in existing:
                    flds.append({
                        "name": c.upper(),
                        "expression": {"dialects": [{"dialect": dialect, "expression": c}]},
                    })

        if model_name:
            model["name"] = model_name
    return yaml.safe_dump(root, sort_keys=False)


# Prefix each bare column with the fact table; SQL function names (followed by "(")
# and already-qualified names (customer.c_name, ORDERS.order_qty) are left alone.
def _qualify_columns(expr, table):
    return re.sub(
        r"(?<![\w.])([A-Za-z_]\w*)(?!\s*\()(?![\w.])",
        lambda m: f"{table}.{m.group(1)}",
        expr,
    )


## Step 5 - Convert the Snowflake Ossie into a Metric View

Read the file, run the shim, run the converter, then align the source names to this catalog and schema.

In [0]:
import yaml
with open(f'{FOLDER}/ossie_from_snowflake.yaml') as fh:
    ossie_v1 = fh.read()

# Moving the file between platforms can double the backslashes in the embedded
# JSON that Snowflake stores in custom_extensions, which makes the YAML invalid.
# If the text does not parse, collapse doubled backslashes and try once more.
# A clean file has none, so this is a no-op on a good file.
try:
    yaml.safe_load(ossie_v1)
except yaml.YAMLError:
    ossie_v1 = ossie_v1.replace('\\\\', '\\')
    yaml.safe_load(ossie_v1)  # raises if still malformed

print(ossie_v1)

version: 0.1.1
semantic_model:
  - name: SALES_SV
    description: Sales star for Ossie interop demo
    datasets:
      - name: CUSTOMERS
        source: DEMOS.SEMANTIC_INTEROP_2.CUSTOMERS
        primary_key:
          - CUSTOMER_ID
        fields:
          - name: CUSTOMER_NAME
            expression:
              dialects:
                - dialect: SNOWFLAKE
                  expression: customer_name
            dimension: {}
          - name: REGION
            expression:
              dialects:
                - dialect: SNOWFLAKE
                  expression: region
            dimension: {}
      - name: ORDERS
        source: DEMOS.SEMANTIC_INTEROP_2.ORDERS
        primary_key:
          - ORDER_ID
        fields:
          - name: ORDER_AMOUNT
            expression:
              dialects:
                - dialect: SNOWFLAKE
                  expression: order_amount
            custom_extensions:
              - vendor_name: SNOWFLAKE
                data: "{\"access_

In [0]:
from ossie_databricks import convert_ossie_to_metric_view, convert_metric_view_to_ossie

converter_ready = snowflake_to_converter(ossie_v1)
mv_yaml = convert_ossie_to_metric_view(converter_ready)
mv_yaml = mv_yaml.replace(SF_NAMESPACE, DBX_NAMESPACE)   # no-op when names match
print(mv_yaml)

version: '1.1'
source: demos.semantic_interop_2.ORDERS
comment: Sales star for Ossie interop demo
joins:
- name: CUSTOMERS
  source: demos.semantic_interop_2.CUSTOMERS
  using:
  - CUSTOMER_ID
  rely:
    at_most_one_match: true
dimensions:
- name: CUSTOMER_NAME
  expr: CUSTOMERS.customer_name
- name: REGION
  expr: CUSTOMERS.region
measures:
- name: ORDER_COUNT
  expr: COUNT(order_id)
- name: TOTAL_ORDER_AMOUNT
  expr: SUM(order_amount)



/local_disk0/.ephemeral_nfs/envs/pythonEnv-31636e3d-5eca-4658-b0aa-99eaf70a05cd/lib/python3.12/site-packages/ossie_databricks/ossie_to_metric_view.py:54: UserWarning: [dataset 'CUSTOMERS'] primary_key/unique_keys not stored as columns; used to set rely.at_most_one_match on a matching many_to_one join where applicable
  warnings.warn(f"[{scope}] {msg}")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-31636e3d-5eca-4658-b0aa-99eaf70a05cd/lib/python3.12/site-packages/ossie_databricks/ossie_to_metric_view.py:54: UserWarning: [dataset 'ORDERS'] primary_key/unique_keys not stored as columns; used to set rely.at_most_one_match on a matching many_to_one join where applicable
  warnings.warn(f"[{scope}] {msg}")


## Step 6 - Create and query the Metric View

Metric-view queries wrap each measure in `MEASURE(...)`. Expected result: EAST 750/5, WEST 700/5.

In [0]:
def create_metric_view(fqname, yaml_body):
    spark.sql('CREATE OR REPLACE VIEW ' + fqname + ' WITH METRICS LANGUAGE YAML AS $$\n' + yaml_body + '\n$$')

create_metric_view(METRIC_VIEW, mv_yaml)
print('created', METRIC_VIEW)

created demos.semantic_interop_2.sales_metric_view


In [0]:
%sql
SELECT region,
       MEASURE(order_count) AS order_count,
       MEASURE(total_order_amount) AS total_order_amount
FROM demos.semantic_interop_2.sales_metric_view
GROUP BY region ORDER BY region

region,order_count,total_order_amount
EAST,5,750
WEST,5,700


## Step 7 - Add a new measure: TOTAL_QUANTITY

Add `TOTAL_QUANTITY = SUM(order_qty)`. Two ways to do it:

- Run the cell below to add it in code.
- Or add it in the Metric View editor UI, then skip the cell below.

Either way, the next step reads the deployed view, so the return trip picks up the
new measure regardless of how you added it.

In [0]:
import yaml
mv = yaml.safe_load(mv_yaml)
mv.setdefault('measures', []).append({'name': 'TOTAL_QUANTITY', 'expr': 'SUM(order_qty)'})
create_metric_view(METRIC_VIEW, yaml.safe_dump(mv, sort_keys=False))
print('added TOTAL_QUANTITY')

added TOTAL_QUANTITY


In [0]:
%sql
SELECT region,
       MEASURE(total_quantity) AS total_quantity,
       MEASURE(total_order_amount) AS total_order_amount,
       MEASURE(order_count) AS order_count
FROM demos.semantic_interop_2.sales_metric_view
GROUP BY region ORDER BY region

region,total_quantity,total_order_amount,order_count
EAST,12,750,5
WEST,11,700,5


## Step 8 - Convert the Metric View back to Ossie

`SHOW CREATE TABLE` returns the deployed view definition. The helper pulls the YAML
out of it, so this reads the real view including any measure added in the UI. The
reverse shim resets the version to 0.1.1, relabels the dialect, rebuilds the fact
columns, and names the model `SALES_SV_V2` for the Snowflake side.

In [0]:
def get_metric_view_yaml(metric_view_name):
    ddl = spark.sql(f'SHOW CREATE TABLE {metric_view_name}').collect()[0][0]
    start = ddl.index('$') + 2
    end = ddl.index('$', start)
    return ddl[start:end].strip()

mv_yaml_v2 = get_metric_view_yaml(METRIC_VIEW)
print(mv_yaml_v2)

version: 1.1

source: demos.semantic_interop_2.ORDERS

joins:
  - name: CUSTOMERS
    source: demos.semantic_interop_2.CUSTOMERS
    using:
      - CUSTOMER_ID
    rely:
      at_most_one_match: true

comment: Sales star for Ossie interop demo

dimensions:
  - name: CUSTOMER_NAME
    expr: CUSTOMERS.customer_name

  - name: REGION
    expr: CUSTOMERS.region

measures:
  - name: ORDER_COUNT
    expr: COUNT(order_id)

  - name: TOTAL_ORDER_AMOUNT
    expr: SUM(order_amount)

  - name: TOTAL_QUANTITY
    expr: SUM(order_qty)


In [0]:
ossie_out = convert_metric_view_to_ossie(mv_yaml_v2)
ossie_out = ossie_out.replace(DBX_NAMESPACE, SF_NAMESPACE)   # align sources back to Snowflake
ossie_v2 = converter_to_snowflake(ossie_out, model_name='SALES_SV_V2')
with open(f'{FOLDER}/ossie_from_databricks.yaml', 'w') as fh:
    fh.write(ossie_v2)
print(ossie_v2)
print('\nWrote ossie_from_databricks.yaml. Download it and upload to the Snowflake stage for notebook 3.')

version: 0.1.1
semantic_model:
- name: SALES_SV_V2
  description: Sales star for Ossie interop demo
  datasets:
  - name: ORDERS
    source: DEMOS.SEMANTIC_INTEROP_2.ORDERS
    fields:
    - name: ORDER_ID
      expression:
        dialects:
        - dialect: SNOWFLAKE
          expression: order_id
    - name: ORDER_AMOUNT
      expression:
        dialects:
        - dialect: SNOWFLAKE
          expression: order_amount
    - name: ORDER_QTY
      expression:
        dialects:
        - dialect: SNOWFLAKE
          expression: order_qty
  - name: CUSTOMERS
    source: DEMOS.SEMANTIC_INTEROP_2.CUSTOMERS
    unique_keys:
    - - CUSTOMER_ID
    fields:
    - name: CUSTOMER_NAME
      expression:
        dialects:
        - dialect: SNOWFLAKE
          expression: customer_name
      dimension: {}
    - name: REGION
      expression:
        dialects:
        - dialect: SNOWFLAKE
          expression: region
      dimension: {}
  relationships:
  - name: ORDERS_to_CUSTOMERS
    from: O